# 텍스트 3종 임베딩 만들기와 눈으로 보기
세 텍스트(원본 캡션, 사물, 맥락 라벨)를 CLIP과 MPNet으로 임베딩한다. 구성 규칙은 Notion 「텍스트 3종의 임베딩 구성 설계」와 저장소의 `src/labeling/text_variants.py`.
- 주 방식 P1: 단위별 임베딩 후 평균 (원본 캡션 5개 / 사물 이름 / 맥락 라벨 구)
- 변형 V2: 한 번에 임베딩 (원본은 캡션 5개를 이음, 사물은 이름을 이은 문자열, 맥락은 라벨을 항목 순서로 이은 문자열)
- 변형 V3: 맥락만, 항목 안에서만 평균한 6블록

Drive의 `fmri_capstone/colab_bundle/`에 `captions.jsonl.gz`, `labels_pass0.jsonl.gz`, `labels_pass9.jsonl.gz`, `stim_info.jsonl.gz`가 있어야 한다.
먼저 `N_LIMIT = 600`으로 빠르게 돌려 오류가 없는지 보고, 문제없으면 `None`으로 바꿔 전체를 돌린다.

In [ ]:
N_LIMIT = 600      # 시험 실행. 전체는 None
VIZ_MODEL = 'MPNet'   # 지도와 이웃 그림에 쓸 모델
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fmri_capstone'
!rm -rf /content/repo && git clone -q https://github.com/geon-2/fmri-imagery-context /content/repo
!pip -q install sentence-transformers

In [ ]:
import sys, gzip, json, re, io, os, collections
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/repo/src/labeling')
import text_variants as tv
B = BASE + '/colab_bundle'
cap = {r['cocoId']: r for r in map(json.loads, gzip.open(B + '/captions.jsonl.gz', 'rt'))}
lab = {r['cocoId']: r for r in map(json.loads, gzip.open(B + '/labels_pass0.jsonl.gz', 'rt'))}
stim = {r['cocoId']: r for r in map(json.loads, gzip.open(B + '/stim_info.jsonl.gz', 'rt'))}
ids = [c for c in cap if c in lab]
rng = np.random.default_rng(0)
if N_LIMIT: ids = sorted(rng.choice(ids, N_LIMIT, replace=False).tolist())
print('맥락 라벨이 있는 이미지', len(ids))
texts = {
    'orig_units': [tv.original_units(cap[c]['captions']) for c in ids],
    'orig_joined': [tv.original_joined(cap[c]['captions']) for c in ids],
    'obj_units': [tv.object_names(cap[c]['object_only']) for c in ids],
    'obj_joined': [tv.object_joined(cap[c]['object_only']) for c in ids],
    'ctx_units': [tv.context_units(lab[c]['labels']) for c in ids],
    'ctx_joined': [tv.context_joined(lab[c]['labels']) for c in ids],
}
i = 3
print('원본 :', texts['orig_joined'][i]); print('사물 :', texts['obj_joined'][i]); print('맥락 :', texts['ctx_joined'][i])

In [ ]:
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
class Emb:
    def __init__(self, name):
        self.name = name
        if name == 'CLIP':
            from transformers import CLIPModel, CLIPTokenizer
            mid = 'openai/clip-vit-large-patch14'
            self.tok = CLIPTokenizer.from_pretrained(mid); self.m = CLIPModel.from_pretrained(mid).to(dev).half().eval(); self.maxlen = 77
        else:
            from sentence_transformers import SentenceTransformer
            self.st = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=dev); self.tok = self.st.tokenizer; self.maxlen = self.st.max_seq_length
    def ntok(self, texts):
        return np.array([len(self.tok(t)['input_ids']) for t in texts])
    def encode(self, texts, bs=256):
        texts = list(texts); out = []
        for k in range(0, len(texts), bs):
            b = texts[k:k + bs]
            if self.name == 'CLIP':
                t = self.tok(b, padding=True, truncation=True, max_length=77, return_tensors='pt').to(dev)
                with torch.no_grad(): e = self.m.text_projection(self.m.text_model(**t).pooler_output)
                e = torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy()
            else:
                e = self.st.encode(b, batch_size=bs, normalize_embeddings=True, show_progress_bar=False)
            out.append(e)
        return np.concatenate(out).astype(np.float32)

def l2(x): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-9)

def embed_units(emb, unit_lists):
    uniq = sorted({u for us in unit_lists for u in us}); ix = {u: i for i, u in enumerate(uniq)}
    E = emb.encode(uniq)
    out = np.zeros((len(unit_lists), E.shape[1]), np.float32)
    for i, us in enumerate(unit_lists):
        if us: out[i] = E[[ix[u] for u in us]].mean(0)
    return l2(out)

def embed_blocks(emb, labels_list):
    by = [tv.context_by_dim(l) for l in labels_list]
    uniq = sorted({p for b in by for d in tv.DIMS for p in b[d]}); ix = {u: i for i, u in enumerate(uniq)}
    E = emb.encode(uniq)
    out = np.zeros((len(by), len(tv.DIMS), E.shape[1]), np.float32); mask = np.zeros((len(by), len(tv.DIMS)), bool)
    for i, b in enumerate(by):
        for j, d in enumerate(tv.DIMS):
            if b[d]: out[i, j] = l2(E[[ix[p] for p in b[d]]].mean(0)); mask[i, j] = True
    return out, mask

## 1. 텍스트 길이 (토큰 수). 한 번에 임베딩하는 변형 V2에서 CLIP은 77토큰을 넘으면 잘린다

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex='row')
names = {'orig_joined': '원본(캡션 5개를 이음)', 'obj_joined': '사물(이름 문자열)', 'ctx_joined': '맥락(라벨을 이음)'}
tokcount = {}
EM = {}
for r, mname in enumerate(['CLIP', 'MPNet']):
    EM[mname] = Emb(mname)
    for c, (k, title) in enumerate(names.items()):
        n = EM[mname].ntok(texts[k]); tokcount[(mname, k)] = n
        axes[r, c].hist(n, bins=40, color='#4c72b0')
        axes[r, c].axvline(EM[mname].maxlen, color='red', ls='--')
        axes[r, c].set_title(f'{mname} | {title}\n평균 {n.mean():.0f}, 한도({EM[mname].maxlen}) 초과 {np.mean(n > EM[mname].maxlen):.1%}')
        axes[r, c].set_xlabel('token 수')
plt.tight_layout(); os.makedirs(BASE + '/figures_v2', exist_ok=True); plt.savefig(BASE + '/figures_v2/token_lengths.png', dpi=150); plt.show()

## 2. 임베딩 만들기와 저장

In [ ]:
REP = {}
OUT = BASE + '/embeddings_v2'; os.makedirs(OUT, exist_ok=True)
json.dump(ids, open(OUT + '/ctx_ids.json', 'w'))
for mname, em in EM.items():
    R = {}
    R['orig_P1'] = embed_units(em, texts['orig_units'])
    R['orig_V2'] = em.encode(texts['orig_joined'])
    R['obj_P1'] = embed_units(em, texts['obj_units'])
    R['obj_V2'] = em.encode(texts['obj_joined'])
    R['ctx_P1'] = embed_units(em, texts['ctx_units'])
    R['ctx_V2'] = em.encode(texts['ctx_joined'])
    blocks, mask = embed_blocks(em, [lab[c]['labels'] for c in ids])
    REP[mname] = R
    suffix = '_small' if N_LIMIT else ''
    for k, v in R.items(): np.save(f'{OUT}/{mname}__{k}{suffix}.npy', v.astype(np.float16))
    np.savez_compressed(f'{OUT}/{mname}__ctx_V3{suffix}.npz', blocks=blocks.astype(np.float16), mask=mask)
    REP[mname]['_blocks'] = (blocks, mask)
    print(mname, {k: v.shape for k, v in R.items() if k != '_blocks'}, '완료')

## 3. 표현들이 서로 얼마나 비슷한가 (RDM 상관)
이미지 1,000장 사이의 거리 구조(RDM)를 표현마다 만들고 서로의 Spearman 상관을 본다. 값이 클수록 두 표현이 이미지 사이의 관계를 비슷하게 본다.
V3는 notes를 뺀 5개 항목 RDM의 평균이다(비어 있는 implied_event는 모든 이미지와 거리 1로 취급).

In [ ]:
from scipy.stats import rankdata
n_rdm = min(1000, len(ids)); sub = np.sort(rng.choice(len(ids), n_rdm, replace=False)); iu = np.triu_indices(n_rdm, 1)
def rdm(X): return 1 - X @ X.T
def vec(D): return rankdata(D[iu])
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
labels_h = ['orig_P1', 'orig_V2', 'obj_P1', 'obj_V2', 'ctx_P1', 'ctx_V2', 'ctx_V3']
for ax, mname in zip(axes, ['CLIP', 'MPNet']):
    R = REP[mname]; V = {}
    for k in labels_h[:-1]: V[k] = vec(rdm(R[k][sub]))
    blocks, mask = R['_blocks']
    V['ctx_V3'] = vec(np.mean([rdm(blocks[sub, j]) for j in range(5)], axis=0))
    M = np.corrcoef(np.array([V[k] for k in labels_h]))
    im = ax.imshow(M, vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(range(7)); ax.set_xticklabels(labels_h, rotation=45, ha='right'); ax.set_yticks(range(7)); ax.set_yticklabels(labels_h)
    for a in range(7):
        for b in range(7): ax.text(b, a, f'{M[a, b]:.2f}', ha='center', va='center', color='white' if M[a, b] < 0.6 else 'black', fontsize=9)
    ax.set_title(f'{mname}: 표현 사이의 RDM 상관')
plt.colorbar(im, ax=axes, shrink=0.8); plt.savefig(BASE + '/figures_v2/rdm_correlation.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. 몇 개의 축이 분산을 설명하나 (PCA 누적 분산). 낮은 차원에 몰려 있을수록 적은 축으로 표현된다

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, mname in zip(axes, ['CLIP', 'MPNet']):
    for k in ['orig_P1', 'obj_P1', 'ctx_P1', 'orig_V2', 'obj_V2', 'ctx_V2']:
        X = REP[mname][k]; X = X - X.mean(0)
        s = np.linalg.svd(X, compute_uv=False) ** 2; cum = np.cumsum(s) / s.sum()
        k90 = int(np.searchsorted(cum, 0.9)) + 1
        ax.plot(range(1, 129), cum[:128], ls='-' if k.endswith('P1') else '--', label=f'{k} (90%: {k90}축)')
    ax.set_xlabel('PC 수'); ax.set_ylabel('누적 분산 비율'); ax.set_title(mname); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(BASE + '/figures_v2/pca_cumulative.png', dpi=150); plt.show()

## 5. 임베딩 공간의 모양 (t-SNE, 점 색은 환경 라벨의 실내/실외)

In [ ]:
from sklearn.manifold import TSNE
def env_flag(c):
    t = ' '.join(v[0].lower() for v in lab[c]['labels']['environment'])
    ind, out = bool(re.search(r'\bindoor|\binterior', t)), bool(re.search(r'\boutdoor|\bexterior', t))
    return 'indoor' if ind and not out else 'outdoor' if out and not ind else 'other'
flags = np.array([env_flag(c) for c in ids])
n_t = min(2500, len(ids)); st = np.sort(rng.choice(len(ids), n_t, replace=False))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
col = {'indoor': '#d62728', 'outdoor': '#2ca02c', 'other': '#bbbbbb'}
for ax, k in zip(axes, ['orig_P1', 'obj_P1', 'ctx_P1']):
    xy = TSNE(2, init='pca', perplexity=30, random_state=0).fit_transform(REP[VIZ_MODEL][k][st])
    for f in col: ax.scatter(*xy[flags[st] == f].T, s=4, c=col[f], label=f)
    ax.set_title(f'{VIZ_MODEL} | {k}'); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=3)
plt.tight_layout(); plt.savefig(BASE + '/figures_v2/tsne.png', dpi=150); plt.show()

## 6. 이웃 비교: 같은 이미지를 기준으로 세 표현이 각각 어떤 이미지를 가깝다고 보나 (왼쪽 첫 칸이 기준 이미지)

In [ ]:
import requests
from PIL import Image
def load_img(c):
    s = stim[c]
    try:
        im = Image.open(io.BytesIO(requests.get(f"http://images.cocodataset.org/{s['cocoSplit']}/{int(c):012d}.jpg", timeout=20).content)).convert('RGB')
        im.thumbnail((220, 220)); return im
    except Exception:
        return Image.new('RGB', (220, 150), 'lightgray')
def neighbors(k, q, topn=5):
    X = REP[VIZ_MODEL][k]; sim = X @ X[q]; sim[q] = -9
    return np.argsort(-sim)[:topn]
reps = ['orig_P1', 'obj_P1', 'ctx_P1']
for q in rng.choice(len(ids), 3, replace=False):
    fig, axes = plt.subplots(3, 6, figsize=(18, 8.5))
    for r, k in enumerate(reps):
        order = [q] + list(neighbors(k, q))
        for cidx, j in enumerate(order):
            ax = axes[r, cidx]; ax.imshow(load_img(ids[j])); ax.axis('off')
            ax.set_title(('기준 ' if cidx == 0 else '') + (k if cidx == 0 else ''), fontsize=9)
    plt.suptitle(f'{VIZ_MODEL} | 기준 이미지 {ids[q]}: ' + texts['orig_units'][q][0][:80] + ' / 맥락: ' + '; '.join(texts['ctx_units'][q][:3])[:90], fontsize=10)
    plt.tight_layout(); plt.show()

## 보는 법
- 3번(RDM 상관): 원본과 맥락이 비슷하게 이미지 사이를 보는가, 사물은 얼마나 다른가. P1과 V2가 서로 얼마나 달라지는가(구성에 따라 표현이 많이 변하면 민감도 분석이 중요해진다).
- 4번(PCA): 낮은 차원에 몰려 있는 표현일수록 적은 축으로 설명된다. 맥락이 사물보다 더 많은 축이 필요한지 본다.
- 5번(t-SNE): 실내/실외가 어떻게 갈라지는가. 맥락이 사물보다 더 뚜렷하게 모이는지.
- 6번(이웃): 사물은 같은 물체가 있는 이미지를, 맥락은 같은 분위기나 상황의 이미지를 고르는 경향이 보이는지.
- 1번(토큰): V2에서 CLIP이 맥락 문자열을 얼마나 자르는지.